In [24]:
import pandas as pd

transactions = pd.read_csv("../data/phase1/transactions.csv")
transactions.head()

,transaction_id,customer_id,product_id,purchase_date,quantity,unit_price,total_price
0,T0000001,C02389,P023096,24-06-2026,1,349.0,349.0
1,T0000002,C00130,P012708,27-05-2025,1,62.5,62.5
2,T0000003,C00639,P013070,27-08-2025,1,342.0,342.0
3,T0000004,C02976,P024212,23-05-2025,1,30.0,30.0
4,T0000005,C01630,P024501,23-01-2026,2,162.0,324.0


In [25]:
interaction_matrix = transactions.pivot_table(
    index = "customer_id",
    columns = "product_id",
    values = "quantity",
    aggfunc = "sum",
    fill_value = 0
)
print("Interaction matrix shape:", interaction_matrix.shape)

Interaction matrix shape: (3000, 27145)


In [26]:
binary_interaction = (interaction_matrix>0).astype(int)
print("Binary interaction matrix shape:", binary_interaction.shape)

Binary interaction matrix shape: (3000, 27145)


In [27]:
from sklearn.metrics.pairwise import cosine_similarity
customer_similarity = cosine_similarity(binary_interaction)
print("Customer similarity matrix shape:", customer_similarity.shape)

Customer similarity matrix shape: (3000, 3000)


In [28]:
import numpy as np
def find_similar_customers(customer_id, n=5):
    customer_index = interaction_matrix.index.get_loc(customer_id)
    similarity_scores = customer_similarity[customer_index]
    similar_indices = np.argsort(similarity_scores)[::-1]
    similar_customers = [interaction_matrix.index[i] for i in similar_indices if interaction_matrix.index[i] != customer_id][:n]
    return similar_customers

In [29]:
interaction_matrix.index[:10].to_list()

['C00001',
 'C00002',
 'C00003',
 'C00004',
 'C00005',
 'C00006',
 'C00007',
 'C00008',
 'C00009',
 'C00010']

In [30]:
find_similar_customers("C00001", n=5)

['C02626', 'C01218', 'C00418', 'C02131', 'C01208']

In [31]:
interactions = pd.read_csv("../data/phase2/customer_product_interactions.csv")

In [32]:
def collaborative_recommend(customer_id, n=5):
    similar_customers = find_similar_customers(customer_id, n=10)
    purchased = set(interactions[interactions["customer_id"] == customer_id]["product_id"])
    candidate_products = interactions[interactions["customer_id"].isin(similar_customers)]
    candidate_products = candidate_products[~candidate_products["product_id"].isin(purchased)]
    recommendations = (
        candidate_products
        .groupby("product_id").size().sort_values(ascending=False).head(n).reset_index(name="similar_customer_count")
    ) 
    return recommendations

In [33]:
master_products = pd.read_csv("../data/phase1/master_products.csv")

In [34]:
collab_recommendations = collaborative_recommend(
    "C00001", n=5
)
collab_recommendations = collab_recommendations.merge(
    master_products[["product_id", "product_name", "category", "brand", "price", "rating"]], on = "product_id", how = "left"
)
collab_recommendations

,product_id,similar_customer_count,product_name,category,brand,price,rating
0,P010637,2,Water Juice Glass Set - Salsa Hi Ball,"Kitchen, Garden & Pets",Ocean,679.0,3.0
1,P014558,2,"Amla - Anti Oxidant, 500mg",Beauty & Hygiene,Sri Sri Tattva,130.0,NaN
2,P025265,2,Skin Brightening Face Sheet Mask,Beauty & Hygiene,Organic Harvest,99.0,4.2
3,P018485,1,Vitamin Bars - Fresh Orange,Gourmet & World Food,Flat Tummies,290.0,NaN
4,P019044,1,Galata Dof Juice/ Water Glass,"Kitchen, Garden & Pets",Lyra,399.0,4.0


In [35]:
collab_recommendations.to_csv(
    "../models/collaborative_recommendations.csv", index=False
)

In [36]:
collab_recommendations.shape

(5, 7)

In [38]:
def collaborative_candidates(customer_id, n_similar=20, n_candidates=50):
    customer_index=interaction_matrix.index.get_loc(customer_id)
    similarity_scores = customer_similarity[customer_index]
    similar_indices = np.argsort(similarity_scores)[::-1]
    candidates={}
    for i in similar_indices:
        similar_customers = interaction_matrix.index[i]
        if similar_customers == customer_id:
            continue
        similarity = similarity_scores[i]
        customer_products = interaction_matrix.loc[similar_customers]
        purchased_products = customer_products[customer_products>0]
        for product_id, quantity in purchased_products.items():
            if interaction_matrix.loc[customer_id, product_id]>0:
                continue
            score = similarity*quantity
            candidates[product_id] = (candidates.get(product_id, 0) + score)
        if len(similar_indices) >= n_similar:
            break
    recommendations = (
        pd.DataFrame(
            list(candidates.items()),
            columns=["product_id", "collaborative_score"]
        )
        .sort_values(
            "collaborative_score",
            ascending=False
        )
        .head(n_candidates)
    )
    return recommendations

In [39]:
collab_candidates = collaborative_candidates(
    "C00001",
    n_similar=20,
    n_candidates=50
)
collab_candidates.head(10)

,product_id,collaborative_score
26,P026388,0.155126
10,P010637,0.103418
15,P014337,0.103418
29,P026457,0.103418
20,P017394,0.103418
12,P011377,0.103418
23,P022821,0.103418
0,P000801,0.051709
21,P019543,0.051709
22,P022417,0.051709


In [41]:
collab_candidates = (
    collab_candidates
    .merge(
        master_products[
            ["product_id", "product_name", "category", "brand", "price", "rating"]
        ], on="product_id", how="left"
    )
)
collab_candidates

,product_id,collaborative_score,product_name,category,brand,price,rating
0,P026388,0.155126,"Shower Sponge - Lemon & White, Colour May Vary",Beauty & Hygiene,Panache,159.20,4.0
1,P010637,0.103418,Water Juice Glass Set - Salsa Hi Ball,"Kitchen, Garden & Pets",Ocean,679.00,3.0
2,P014337,0.103418,Bio Aloevera Lotion - 30 Spf Sunscreen For Nor...,Beauty & Hygiene,BIOTIQUE,188.00,4.0
3,P026457,0.103418,Pickle - Garlic,Snacks & Branded Foods,Nirapara,135.00,3.7
4,P017394,0.103418,So Soft 3 Ply - Toilet Tissue Rolls,Cleaning & Household,Origami,310.00,4.2
5,P011377,0.103418,"Dry Dog Food - PRO, Expert Nutrition for Activ...","Kitchen, Garden & Pets",Pedigree,2700.00,4.8
6,P022821,0.103418,Large Period Cup For Medium Or Heavy Flow - Po...,Beauty & Hygiene,ezy,439.20,3.7
7,P000801,0.051709,Premium - 3 Follow-Up Formula,Baby Care,Dexolac,595.00,4.5
8,P019543,0.051709,D-Tan Scrub For Men,Beauty & Hygiene,QRAA,263.00,4.2
9,P022417,0.051709,Chalk Gold - For Cockroaches,Cleaning & Household,Krazy Lines,25.00,3.6


In [42]:
purchased_products = set(
    interactions[
        interactions["customer_id"] == "C00001"]["product_id"]
    
)
collab_candidates = collab_candidates[
    ~collab_candidates["product_id"].isin(purchased_products)
].copy()
collab_candidates.head(10)

,product_id,collaborative_score,product_name,category,brand,price,rating
0,P026388,0.155126,"Shower Sponge - Lemon & White, Colour May Vary",Beauty & Hygiene,Panache,159.2,4.0
1,P010637,0.103418,Water Juice Glass Set - Salsa Hi Ball,"Kitchen, Garden & Pets",Ocean,679.0,3.0
2,P014337,0.103418,Bio Aloevera Lotion - 30 Spf Sunscreen For Nor...,Beauty & Hygiene,BIOTIQUE,188.0,4.0
3,P026457,0.103418,Pickle - Garlic,Snacks & Branded Foods,Nirapara,135.0,3.7
4,P017394,0.103418,So Soft 3 Ply - Toilet Tissue Rolls,Cleaning & Household,Origami,310.0,4.2
5,P011377,0.103418,"Dry Dog Food - PRO, Expert Nutrition for Activ...","Kitchen, Garden & Pets",Pedigree,2700.0,4.8
6,P022821,0.103418,Large Period Cup For Medium Or Heavy Flow - Po...,Beauty & Hygiene,ezy,439.2,3.7
7,P000801,0.051709,Premium - 3 Follow-Up Formula,Baby Care,Dexolac,595.0,4.5
8,P019543,0.051709,D-Tan Scrub For Men,Beauty & Hygiene,QRAA,263.0,4.2
9,P022417,0.051709,Chalk Gold - For Cockroaches,Cleaning & Household,Krazy Lines,25.0,3.6


In [43]:
collab_candidates.to_csv("../models/collaborative_candidates_C00001.csv", index=False)